# 04 — The Precursor Test

**Do topological signatures systematically precede breakthroughs?**

This is the hypothesis test. For each cataloged breakthrough, we compare the
topological metrics (β₁, persistence entropy) in the years *before* the
breakthrough against a null model — the same CPC section pair at times when
no breakthrough occurred.

Key method: **Superposed epoch analysis** — align all breakthroughs at t=0
and average the signal. If topology is a precursor, we should see a systematic
rise before t=0 that the null model doesn't exhibit.

---

In [1]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm

from src.utils import DATA_DIR, get_logger
from src.breakthroughs import load_breakthroughs, get_precursor_window
from src.topology import compute_all_priority_pairs, PRIORITY_PAIRS
from src.nullmodel import matched_null, random_cpc_pair_baseline, superposed_epoch
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, year_axis, confidence_band

set_style()
logger = get_logger('nb04')

In [2]:
# %% Load data and breakthroughs
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

# Ensure citing_year column exists (required by new topology module)
citations['citing_year'] = pd.to_datetime(citations['citing_date']).dt.year
citations.drop(columns=['citing_date'], inplace=True)  # Free ~1GB RAM

breakthroughs = load_breakthroughs()
print(f'Breakthroughs: {len(breakthroughs)}')

2026-03-19 22:03:09 | src.breakthroughs        | INFO    | Loaded 34 breakthroughs from 8 files


Breakthroughs: 34


In [3]:
# %% Load topology results from Notebook 02 (re-compute with caching if needed)
import gc

pair_results = compute_all_priority_pairs(
    citations, cpc_map,
    cache_dir=str(DATA_DIR / 'topology_cache'),
    start_year=1985,
    end_year=2023,
)

# Build topology_results dict keyed by both "AxC" and "A_C" formats
topology_results = {}
for pair, group in pair_results.groupby('section_pair'):
    topology_results[pair] = group.reset_index(drop=True)
    # Also add underscore-format key for backward compat with breakthrough lookups
    underscore_key = pair.replace('x', '_')
    topology_results[underscore_key] = group.reset_index(drop=True)

print(f'Total topology results: {len(pair_results["section_pair"].unique())} CPC pairs')
gc.collect()

Total topology results: 10 CPC pairs


0

## Compute Matched Null Models

For each breakthrough, generate null topology measurements from the same
CPC pair but in non-breakthrough time periods.

In [4]:
# %% Generate matched null models
# Load from cache where available; skip breakthroughs that need fresh computation
# (non-priority CPC pairs are too slow on 16GB — 26/34 is sufficient for statistics)
import pickle

null_results = {}
skipped = []

for bt in tqdm(breakthroughs, desc='Loading null models'):
    bt_name = bt.name.replace(" ", "_").replace("/", "_").lower()[:30]
    # Try cached n=100 first, then n=500
    cache_100 = DATA_DIR / "null_cache" / f"matched_{bt_name}_n100_s42_cocite.pkl"
    cache_500 = DATA_DIR / "null_cache" / f"matched_{bt_name}_n500_s42_cocite.pkl"
    
    if cache_100.exists():
        with open(cache_100, "rb") as f:
            null_results[bt.name] = pickle.load(f)
    elif cache_500.exists():
        with open(cache_500, "rb") as f:
            null_results[bt.name] = pickle.load(f)
    else:
        skipped.append(bt.name)

print(f'Null models loaded for {len(null_results)} breakthroughs')
if skipped:
    print(f'Skipped (no cache, non-priority CPC pairs): {skipped}')

Loading null models:   0%|          | 0/34 [00:00<?, ?it/s]

Loading null models: 100%|██████████| 34/34 [00:00<00:00, 1009.71it/s]

Null models loaded for 26 breakthroughs
Skipped (no cache, non-priority CPC pairs): ['EUV Lithography', 'CAR-T Cell Therapy', 'MapReduce Distributed Computing', 'Graphene Production', 'Multi-Touch Interface (iPhone)', '4G LTE Wireless Standard', 'CRISPR-Cas9 Gene Editing', 'mRNA Vaccine Platform']


## Pre-Breakthrough vs. Null Topology

For each breakthrough: extract β₁ in the 10 years before filing,
compare against the matched null distribution.

In [5]:
# %% Extract pre-breakthrough metrics and compute z-scores
# Match breakthroughs to ANY topology pair containing their CPC section(s)
comparison = []

for bt in breakthroughs:
    sections = bt.cpc_sections if len(bt.cpc_sections) >= 1 else []
    if not sections:
        continue
    
    # Find all topology pairs containing at least one of this breakthrough's sections
    matching_topos = []
    for key, topo in topology_results.items():
        if 'x' not in key:
            continue
        sec_a, sec_b = key.split('x')
        if any(s in [sec_a, sec_b] for s in sections):
            matching_topos.append(topo)
    
    if not matching_topos:
        continue
    
    year_col = 'window_end'
    start, end = get_precursor_window(bt, years_before=10)
    
    pre_beta1_vals = []
    pre_pe_vals = []
    for topo in matching_topos:
        if year_col not in topo.columns:
            continue
        pre = topo[(topo[year_col] >= start) & (topo[year_col] <= end)]
        if len(pre) > 0 and 'beta_1' in pre.columns:
            pre_beta1_vals.append(pre['beta_1'].mean())
            pre_pe_vals.append(pre['persistence_entropy'].mean())
    
    if not pre_beta1_vals:
        continue
    
    pre_beta1 = np.mean(pre_beta1_vals)
    pre_pe = np.mean(pre_pe_vals)
    
    null = null_results.get(bt.name)
    if null is None or len(null) == 0:
        continue
    
    null_beta1_mean = null['beta_1'].mean()
    null_beta1_std = null['beta_1'].std()
    null_pe_mean = null['persistence_entropy'].mean()
    null_pe_std = null['persistence_entropy'].std()
    
    # Use percentile rank instead of z-score when null std is degenerate
    # This avoids division-by-near-zero artifacts
    if null_beta1_std > 1.0:
        z_beta1 = (pre_beta1 - null_beta1_mean) / null_beta1_std
    else:
        # Percentile rank: what fraction of null samples is pre_beta1 above?
        pct = (null['beta_1'] < pre_beta1).mean()
        z_beta1 = stats.norm.ppf(max(0.001, min(0.999, pct)))  # Convert to z-equivalent
    
    if null_pe_std > 0.01:
        z_pe = (pre_pe - null_pe_mean) / null_pe_std
    else:
        pct = (null['persistence_entropy'] < pre_pe).mean()
        z_pe = stats.norm.ppf(max(0.001, min(0.999, pct)))
    
    comparison.append({
        'name': bt.name,
        'category': bt.category,
        'filing_year': bt.filing_year,
        'pre_beta1': pre_beta1,
        'null_beta1_mean': null_beta1_mean,
        'null_beta1_std': null_beta1_std,
        'z_beta1': z_beta1,
        'pre_pe': pre_pe,
        'null_pe_mean': null_pe_mean,
        'null_pe_std': null_pe_std,
        'z_pe': z_pe,
        'n_matching_pairs': len(matching_topos),
    })

comp_df = pd.DataFrame(comparison)
print(f'Breakthroughs with valid comparisons: {len(comp_df)}')
print(f'\nMean z-score (β₁):  {comp_df["z_beta1"].mean():.3f} ± {comp_df["z_beta1"].std():.3f}')
print(f'Mean z-score (PE):   {comp_df["z_pe"].mean():.3f} ± {comp_df["z_pe"].std():.3f}')
print(f'\nPer-breakthrough results:')
print(comp_df[['name', 'filing_year', 'z_beta1', 'z_pe', 'n_matching_pairs']].to_string(index=False))

Breakthroughs with valid comparisons: 14

Mean z-score (β₁):  -2.934 ± 9.447
Mean z-score (PE):   -20.709 ± 28.530

Per-breakthrough results:
                                           name  filing_year    z_beta1       z_pe  n_matching_pairs
             Backpropagation Learning Algorithm         1989  -6.187230 -49.951368                 4
    Fused Deposition Modeling (FDM) 3D Printing         1989  -9.178304 -67.511385                 3
                  Convolutional Neural Networks         1990 -25.344792  -3.090232                 4
           Modern Wind Turbine (Variable Speed)         1990  -6.306026 -47.668411                 1
                    Elliptic Curve Cryptography         1991  12.366785   9.202810                 8
                              CMOS Image Sensor         1993   9.429173   9.007011                 8
                             WiFi (IEEE 802.11)         1993  -6.368155 -49.328581                 5
                       USB Universal Serial Bus   

## Figure 1: Superposed Epoch Plot (THE KEY RESULT)

Align all breakthroughs at t=0 (filing year), average β₁ from -10 to +5 years.
Compare against the null model's 95% confidence interval.

In [6]:
# %% Figure 1: Superposed epoch analysis
epoch = superposed_epoch(
    breakthroughs=breakthroughs,
    topology_results=topology_results,
    metric='beta_1',
    years_before=10,
    years_after=5,
)

# Generate null epoch (using random baseline)
# Use small sample — most will hit topology cache
null_baseline = random_cpc_pair_baseline(
    citations=citations,
    cpc_map=cpc_map,
    n_samples=50,
    seed=42,
)

fig, ax = plt.subplots(figsize=(12, 7))

if len(epoch) > 0:
    ax.plot(epoch['epoch_year'], epoch['mean'], color=PALETTE['red'], linewidth=2.5,
            label='Pre-breakthrough β₁ (mean)', zorder=5)
    ax.fill_between(
        epoch['epoch_year'],
        epoch['mean'] - epoch['std'],
        epoch['mean'] + epoch['std'],
        alpha=0.2, color=PALETTE['red'], label='±1 SD'
    )

# Null band
if len(null_baseline) > 0:
    null_mean = null_baseline['beta_1'].mean()
    null_std = null_baseline['beta_1'].std()
    ax.axhspan(null_mean - 1.96 * null_std, null_mean + 1.96 * null_std,
               alpha=0.1, color=PALETTE['blue'], label='Null 95% CI')
    ax.axhline(null_mean, color=PALETTE['blue'], linestyle='--', alpha=0.5)

ax.axvline(0, color='gray', linestyle=':', alpha=0.7, label='Breakthrough (t=0)')
ax.set_xlabel('Years Relative to Breakthrough Filing')
ax.set_ylabel('β₁ (Mean Across Breakthroughs)')
ax.set_title('Figure 1: Superposed Epoch Analysis — β₁ Before and After Breakthroughs')
ax.legend()
fig.tight_layout()
save_figure(fig, '04_superposed_epoch')

2026-03-19 22:18:07 | src.nullmodel            | WARNING | No topology data for Hydraulic Fracturing with Horizontal Drilling (sections: ['E'])


2026-03-19 22:18:07 | src.nullmodel            | INFO    | Loading cached random baseline


2026-03-19 22:18:07 | timer                    | INFO    | random_cpc_pair_baseline completed in 0.0s


2026-03-19 22:18:08 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_superposed_epoch.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_superposed_epoch.png')

## Figure 2: Individual Breakthrough β₁ Curves

In [7]:
# %% Figure 2: Individual breakthrough curves
selected_bts = [bt for bt in breakthroughs if bt.name in [
    'Lithium-Ion Battery', 'PageRank Algorithm', 
    'Polymerase Chain Reaction (PCR)', 'Recombinant Human Insulin',
    'Selective Laser Sintering (SLS) 3D Printing',
    'GPU General-Purpose Computing (CUDA)', 'WiFi (IEEE 802.11)',
    'OLED Display Technology', 'Convolutional Neural Networks (CNN)',
]]

n_selected = len(selected_bts)
ncols = 3
nrows = (n_selected + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows), sharex=True)
axes = axes.flatten() if n_selected > 1 else [axes]

for i, bt in enumerate(selected_bts):
    ax = axes[i]
    sections = bt.cpc_sections if len(bt.cpc_sections) >= 1 else []
    
    # Find all matching topology pairs (same broad matching as cell 7)
    plotted = False
    for key, topo in topology_results.items():
        if 'x' not in key:
            continue
        sec_a, sec_b = key.split('x')
        if any(s in [sec_a, sec_b] for s in sections):
            year_col = 'window_end' if 'window_end' in topo.columns else 'year'
            ax.plot(topo[year_col], topo['beta_1'], linewidth=1.5, alpha=0.6, label=key)
            plotted = True
    
    if plotted:
        ax.axvline(bt.filing_year, color=PALETTE['red'], linestyle='--', alpha=0.7, label='Filing year')
        ax.axvspan(bt.filing_year - 10, bt.filing_year, alpha=0.05, color=PALETTE['orange'])
    
    ax.set_title(bt.name, fontsize=10)
    ax.set_ylabel('β₁')
    if i == 0:
        ax.legend(fontsize=6, loc='upper right')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Figure 2: β₁ Time Series for Individual Breakthroughs\n(colored by CPC section pair)', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, '04_individual_beta1')

2026-03-19 22:18:10 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_individual_beta1.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_individual_beta1.png')

## Figure 3: Effect Size Distribution

In [8]:
# %% Figure 3: Z-score distribution
if len(comp_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # β₁ z-scores
    ax = axes[0]
    ax.barh(range(len(comp_df)), comp_df['z_beta1'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_beta1']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3, label='p=0.05')
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (β₁)')
    ax.set_title('β₁ Effect Size')
    ax.legend()
    ax.invert_yaxis()

    # PE z-scores
    ax = axes[1]
    ax.barh(range(len(comp_df)), comp_df['z_pe'].values,
            color=[PALETTE['red'] if z > 0 else PALETTE['blue'] for z in comp_df['z_pe']])
    ax.set_yticks(range(len(comp_df)))
    ax.set_yticklabels(comp_df['name'].values, fontsize=7)
    ax.axvline(0, color='gray', linestyle='-', alpha=0.5)
    ax.axvline(1.96, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(-1.96, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Z-score (Persistence Entropy)')
    ax.set_title('Persistence Entropy Effect Size')
    ax.invert_yaxis()

    fig.suptitle('Figure 3: Pre-Breakthrough Topology vs. Null Model', fontsize=14, y=1.02)
    fig.tight_layout()
    save_figure(fig, '04_effect_sizes')

2026-03-19 22:18:11 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_effect_sizes.png


## Statistical Tests

In [9]:
# %% Aggregate statistical tests
if len(comp_df) > 0:
    # One-sample t-test: are z-scores significantly different from 0?
    t_stat_b1, p_val_b1 = stats.ttest_1samp(comp_df['z_beta1'], 0)
    t_stat_pe, p_val_pe = stats.ttest_1samp(comp_df['z_pe'], 0)
    
    print('=== Aggregate Statistical Tests ===')
    print(f'\nβ₁ z-scores: mean={comp_df["z_beta1"].mean():.3f}, t={t_stat_b1:.3f}, p={p_val_b1:.4f}')
    print(f'PE z-scores: mean={comp_df["z_pe"].mean():.3f}, t={t_stat_pe:.3f}, p={p_val_pe:.4f}')
    
    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat_b1, wp_b1 = stats.wilcoxon(comp_df['z_beta1'])
        w_stat_pe, wp_pe = stats.wilcoxon(comp_df['z_pe'])
        print(f'\nWilcoxon β₁: W={w_stat_b1:.1f}, p={wp_b1:.4f}')
        print(f'Wilcoxon PE: W={w_stat_pe:.1f}, p={wp_pe:.4f}')
    except ValueError:
        print('Wilcoxon test requires >20 samples, skipped.')
    
    # Effect size (Cohen's d)
    d_b1 = comp_df['z_beta1'].mean() / comp_df['z_beta1'].std() if comp_df['z_beta1'].std() > 0 else 0
    d_pe = comp_df['z_pe'].mean() / comp_df['z_pe'].std() if comp_df['z_pe'].std() > 0 else 0
    print(f'\nCohen\'s d (β₁): {d_b1:.3f}')
    print(f'Cohen\'s d (PE):  {d_pe:.3f}')
    
    # Interpretation
    if p_val_b1 < 0.05:
        print(f'\n** β₁ is significantly different in pre-breakthrough periods (p={p_val_b1:.4f}) **')
    else:
        print(f'\n** β₁ is NOT significantly different from null (p={p_val_b1:.4f}) **')
        print('   This is a null result: topological loops do not systematically precede breakthroughs.')

=== Aggregate Statistical Tests ===

β₁ z-scores: mean=-2.934, t=-1.162, p=0.2662
PE z-scores: mean=-20.709, t=-2.716, p=0.0176

Wilcoxon β₁: W=39.0, p=0.4263
Wilcoxon PE: W=23.0, p=0.0640

Cohen's d (β₁): -0.311
Cohen's d (PE):  -0.726

** β₁ is NOT significantly different from null (p=0.2662) **
   This is a null result: topological loops do not systematically precede breakthroughs.


## Figure 4: ROC-Like Curve

If we treat high β₁ as a "breakthrough detector", how well does it discriminate?

In [10]:
# %% Figure 4: Detection performance
if len(comp_df) > 0 and len(null_baseline) > 0:
    # Pre-breakthrough β₁ values
    real_values = comp_df['pre_beta1'].values
    null_values = null_baseline['beta_1'].values
    
    # Sweep thresholds
    thresholds = np.linspace(
        min(real_values.min(), null_values.min()),
        max(real_values.max(), null_values.max()),
        100
    )
    
    hit_rates = []
    false_alarm_rates = []
    
    for thresh in thresholds:
        hits = (real_values > thresh).mean()
        false_alarms = (null_values > thresh).mean()
        hit_rates.append(hits)
        false_alarm_rates.append(false_alarms)
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(false_alarm_rates, hit_rates, color=PALETTE['red'], linewidth=2)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC=0.5)')
    
    # Compute AUC
    auc = np.trapz(sorted(hit_rates), sorted(false_alarm_rates))
    ax.set_xlabel('False Alarm Rate')
    ax.set_ylabel('Hit Rate')
    ax.set_title(f'Figure 4: β₁ as Breakthrough Detector (AUC = {abs(auc):.3f})')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    fig.tight_layout()
    save_figure(fig, '04_roc_curve')

2026-03-19 22:18:11 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/04_roc_curve.png


## Summary

**The Precursor Test answers the central question:**
Do topological signatures in the patent citation network systematically
precede technological breakthroughs?

The answer depends on the data. If the superposed epoch shows a clear signal
above the null band, topology is a leading indicator. If not, innovation
is topologically invisible until it arrives — which is itself a meaningful
scientific finding.

Either way, this feeds into Notebook 05 where we test whether topology adds
predictive power beyond simpler metrics.